<a href="https://colab.research.google.com/github/MariaPetrovskaya/LLM-applications-/blob/main/RAG_in_Law/version_for_github_vectorize_law.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⚖️ Law Vectorizer — TK RF Edition

Loads a legal document (**PDF / TXT / Google Doc**), splits it into **article-level
chunks** parsed from section headings (`Статья N.`), generates **512-dim multilingual
embeddings**, and upserts them into **Pinecone** with rich structured metadata.

| Setting | Value |
|---|---|
| Embedding model | `distiluse-base-multilingual-cased-v2` |
| Vector dimension | **512** |
| Similarity metric | cosine |
| Chunk strategy | Article-title boundaries (`Статья N.`) |
| Batch size | Adaptive — scales with article count |
| Input formats | `.pdf`, `.txt`, **`.gdoc`** (Google Docs stub) |

## 1 · Install dependencies

In [ ]:
!pip install -q pinecone sentence-transformers pypdf2 pandas openpyxl chardet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 8.2 MB/s eta 0:00:00


## 2 · Imports & GPU check

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import re
import time
import uuid
import hashlib
from itertools import islice

import numpy as np
import pandas as pd
import torch
import PyPDF2
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec

print(f'GPU available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU name      : {torch.cuda.get_device_name()}')
else:
    print('Running on CPU')

Mounted at /content/drive
GPU available : False
Running on CPU


## 3 · Configuration — edit this cell

In [ ]:
# ── Pinecone ──────────────────────────────────────────────────────────────
PINECONE_API_KEY = '####################################'   # <- replace
INDEX_NAME       = 'legis'
Namespace        = 'new2'
VECTOR_DIM       = 512
METRIC           = 'cosine'
CLOUD            = 'aws'
REGION           = 'us-east-1'

# ── Embedding model (exactly 512 dims, multilingual RU/EN) ────────────────
MODEL_NAME = 'sentence-transformers/distiluse-base-multilingual-cased-v2'

# ── Source document ───────────────────────────────────────────────────────
FILE_PATH = '/content/drive/MyDrive/RAG_in_low/trudkod.gdoc'   # <- replace

# ── Document metadata (stored in every Pinecone vector) ──────────────────
# Matches the full n8n/LangChain Google Drive loader schema exactly:
#   blobType | date_created | department | filename | link | load_date
import os as _os, datetime as _dt

DOC_FILENAME     = _os.path.basename(FILE_PATH)   # auto-derived; override if needed
DOC_DEPARTMENT   = 'legis'                        # <- your Pinecone namespace / collection
DOC_BLOB_TYPE    = 'application/pdf'              # 'application/pdf' | 'text/plain'
DOC_LINK         = ''                             # <- paste Google Drive share link, or leave empty
_now             = _dt.datetime.now(_dt.timezone.utc)
DOC_DATE_CREATED = _now.strftime('%Y-%m-%dT%H:%M:%S.') + f'{_now.microsecond // 1000:03d}Z'
DOC_LOAD_DATE    = _now.strftime('%Y-%m-%d')

# ── Article heading pattern ───────────────────────────────────────────────
import re
ARTICLE_PATTERN = re.compile(r'(?m)^\s*(Статья\s+\d+[\d\.]*\.\s*.+)$')

print('Configuration loaded.')
print(f'  filename     : {DOC_FILENAME}')
print(f'  department   : {DOC_DEPARTMENT}')
print(f'  blobType     : {DOC_BLOB_TYPE}')
print(f'  date_created : {DOC_DATE_CREATED}')
print(f'  load_date    : {DOC_LOAD_DATE}')
print(f'  link         : {DOC_LINK or "(not set)"}')


Configuration loaded.
  filename     : trudkod.gdoc
  department   : legis
  blobType     : application/pdf
  date_created : 2026-05-14T10:02:12.032Z
  load_date    : 2026-05-14
  link         : (not set)


## 4 · Load document (PDF or TXT)

## 4b · Loading `.gdoc` files

A `.gdoc` file on a mounted Google Drive can **not** be opened with `open()`
(raises `OSError: [Errno 95]`). Instead, we read it as **binary via Drive's
FUSE interface** using `os.open()` with low-level flags, or export it through
the authenticated Colab HTTP session.

The function below calls
`https://docs.google.com/document/d/{doc_id}/export?format=txt`
using the token from `google.auth` — no API client, no credentials config,
no extra setup beyond `drive.mount`.

In [ ]:
def _decode_bytes(raw: bytes) -> str:
    """Decode exported bytes, handling UTF-16 BOM and common Cyrillic encodings."""
    if raw[:2] in (b'\xff\xfe', b'\xfe\xff'):
        return raw.decode('utf-16')
    for enc in ('utf-8', 'utf-8-sig', 'windows-1251', 'windows-1252', 'latin-1'):
        try:
            return raw.decode(enc)
        except (UnicodeDecodeError, LookupError):
            continue
    try:
        import chardet
        enc = chardet.detect(raw).get('encoding') or 'utf-8'
        print(f'chardet detected: {enc}')
        return raw.decode(enc)
    except ImportError:
        pass
    return raw.decode('utf-8', errors='replace')


def read_gdoc(gdoc_path: str) -> str:
    """
    Export a Google Doc as plain text via Colab's authenticated session.
    Only requires drive.mount — no API keys or credentials config.

    The .gdoc stub is never opened directly.
    The doc ID is extracted from the filename or found via Drive search.
    """
    import requests
    from google.colab import auth
    from google.auth.transport.requests import Request as GRequest
    import google.auth

    # ── Resolve doc ID from filename (works even without stub access) ────
    import re as _re
    m = _re.search(r'\(([A-Za-z0-9_-]{25,})\)', os.path.basename(gdoc_path))
    if m:
        doc_id = m.group(1)
        print(f'Doc ID from filename: {doc_id}')
    else:
        # Search Drive by document name
        from googleapiclient.discovery import build
        filename_no_ext = os.path.splitext(os.path.basename(gdoc_path))[0]
        auth.authenticate_user()
        creds, _ = google.auth.default(
            scopes=['https://www.googleapis.com/auth/drive.readonly'])
        creds.refresh(GRequest())
        svc  = build('drive', 'v3', credentials=creds)
        safe = filename_no_ext.replace("'", "\\'")
        q    = (f"name='{safe}' and "
                "mimeType='application/vnd.google-apps.document' and trashed=false")
        res  = svc.files().list(q=q, fields='files(id,name)', pageSize=3).execute()
        hits = res.get('files', [])
        if not hits:
            raise FileNotFoundError(
                f'No Google Doc named "{filename_no_ext}" found in Drive.')
        doc_id = hits[0]['id']
        print(f'Found "{hits[0]["name"]}" → {doc_id}')

    # ── Authenticate and export ───────────────────────────────────────────
    auth.authenticate_user()
    creds, _ = google.auth.default(
        scopes=['https://www.googleapis.com/auth/drive.readonly'])
    creds.refresh(GRequest())

    url      = f'https://docs.google.com/document/d/{doc_id}/export?format=txt'
    response = requests.get(url, headers={'Authorization': f'Bearer {creds.token}'})
    response.raise_for_status()

    text = _decode_bytes(response.content)
    print(f'Loaded: {len(text):,} chars')
    return text


print('read_gdoc() ready — only drive.mount() required.')

read_gdoc() ready — only drive.mount() required.


In [ ]:
def read_pdf(path: str) -> str:
    """Extract all text from a PDF file."""
    with open(path, 'rb') as fh:
        reader = PyPDF2.PdfReader(fh)
        pages = [p.extract_text() or '' for p in reader.pages]
    text = '\n'.join(pages)
    print(f'PDF loaded: {len(reader.pages)} pages, {len(text):,} chars')
    return text


def read_txt(path: str) -> str:
    """Read a plain-text file."""
    with open(path, 'r', encoding='utf-8') as fh:
        text = fh.read()
    print(f'TXT loaded: {len(text):,} chars')
    return text


def load_document(path: str) -> str:
    """Load PDF, TXT, or Google Doc (.gdoc) and return plain text."""
    if not os.path.exists(path):
        raise FileNotFoundError(f'File not found: {path}')
    ext = os.path.splitext(path)[1].lower()
    if ext == '.pdf':  return read_pdf(path)
    if ext == '.txt':  return read_txt(path)
    if ext == '.gdoc': return read_gdoc(path)
    raise ValueError(f'Unsupported format: {ext}. Use .pdf, .txt, or .gdoc')


raw_text = load_document(FILE_PATH)
print('\n--- First 500 chars ---')
print(raw_text[:500])

Found "trudkod" → 1J_FpEv6T85Oh_dqcyB5y6Ga57SvYCmpOqPPzXg6qbgo
Loaded: 802,560 chars

--- First 500 chars ---
﻿Трудовой кодекс Российской Федерации от 30 декабря 2001 г. N 197-ФЗ (ТК РФ) (с изменениями и дополнениями)


Принят Государственной Думой 21 декабря 2001 года
Одобрен Советом Федерации 26 декабря 2001 года
Часть первая


Раздел I. Общие положения
Глава 1. Основные начала трудового законодательства


Статья 1. Цели и задачи трудового законодательства
Целями трудового законодательства являются установление государственных гарантий трудовых прав и свобод граждан, создание благоприятны


## 5 · Article-based chunking

Each chunk = one complete article (from its heading to the start of the next).
Article number and title are extracted directly from the heading line.

In [ ]:
def parse_article_title(heading_line: str) -> dict:
    """
    Parse article number and title from a heading.
    Input:  'Статья 255. Отпуска по беременности и родам'
    Output: article_num='255', article_short='Отпуска по...'
    """
    m = re.match(r'Статья\s+(\d+[\d\.]*)\.\s*(.*)', heading_line.strip())
    if not m:
        return {
            'article_num'   : '',
            'article_short' : heading_line.strip(),
        }
    return {
        'article_num'   : m.group(1).rstrip('.'),
        'article_short' : m.group(2).strip(),
    }


def split_by_articles(text: str, pattern=ARTICLE_PATTERN) -> list:
    """
    Split full law text into per-article dicts.
    Falls back to single-chunk if no article headings are found.
    """
    matches = list(pattern.finditer(text))
    if not matches:
        print('WARNING: No article headings found — treating whole text as one chunk.')
        return [{'heading': '', 'full_text': text, 'article_num': '0', 'article_short': ''}]

    chunks = []
    for i, match in enumerate(matches):
        heading = match.group(1).strip()
        start   = match.start()
        end     = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        body    = text[start:end].strip()
        chunks.append({'heading': heading, 'full_text': body,
                       **parse_article_title(heading)})

    print(f'Parsed {len(chunks)} articles.')
    return chunks


article_chunks = split_by_articles(raw_text)

# Preview first two articles
for c in article_chunks[:2]:
    print(f"  [ст. {c['article_num']}] {c['article_short'][:60]}  ({len(c['full_text'])} chars)")


Parsed 528 articles.
  [ст. 1] Цели и задачи трудового законодательства  (1407 chars)
  [ст. 2] Основные принципы правового регулирования трудовых отношений  (3492 chars)


## 6 · Build metadata list (n8n/LangChain schema)

Each chunk gets all six metadata fields that match the Google Drive loader output:
`blobType`, `date_created`, `department`, `filename`, `link`, `load_date`.


In [ ]:
def build_metadata_list(chunks: list) -> list:
    """
    Build records matching the full n8n/LangChain Google Drive → Pinecone loader schema:

        pageContent   — full article text (embedding source)
        metadata:
            blobType      — MIME type of the source file
            date_created  — ISO-8601 UTC timestamp with ms  (e.g. "2026-04-29T16:26:27.148Z")
            department    — Pinecone namespace / collection label
            filename      — original source file name
            link          — Google Drive shareable URL (empty string if not applicable)
            load_date     — ingestion date  (e.g. "2026-04-29")
    """
    records = []
    for chunk in chunks:
        records.append({
            'pageContent': chunk['full_text'],
            'metadata': {
                'blobType'     : DOC_BLOB_TYPE,
                'date_created' : DOC_DATE_CREATED,
                'department'   : DOC_DEPARTMENT,
                'filename'     : DOC_FILENAME,
                'link'         : DOC_LINK,
                'load_date'    : DOC_LOAD_DATE,
            }
        })
    return records


metadata_list  = build_metadata_list(article_chunks)
texts_to_embed = [r['pageContent'] for r in metadata_list]

print(f'Total chunks: {len(metadata_list)}')
print(f'\nSample metadata:')
for k, v in metadata_list[0]['metadata'].items():
    print(f'  {k:15s}: {v}')
print(f'  pageContent  : {metadata_list[0]["pageContent"][:80]}...')


Total chunks: 528

Sample metadata:
  blobType       : application/pdf
  date_created   : 2026-05-14T10:02:12.032Z
  department     : legis
  filename       : trudkod.gdoc
  link           : 
  load_date      : 2026-05-14
  pageContent  : Статья 1. Цели и задачи трудового законодательства
Целями трудового законодател...


## 7 · Load embedding model (512-dim)

In [ ]:
model = SentenceTransformer(MODEL_NAME)

actual_dim = model.get_sentence_embedding_dimension()
print(f'Model        : {MODEL_NAME}')
print(f'Embedding dim: {actual_dim}')

assert actual_dim == VECTOR_DIM, (
    f'Dimension mismatch! Model outputs {actual_dim}, '
    f'but Pinecone index expects {VECTOR_DIM}. '
    'Update MODEL_NAME or VECTOR_DIM in the config cell.'
)

modules.json:   0%|          | 0.00/341 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/610 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/1.58M [00:00<?, ?B/s]

Model        : sentence-transformers/distiluse-base-multilingual-cased-v2
Embedding dim: 512


/tmp/ipykernel_8955/2703598476.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  actual_dim = model.get_sentence_embedding_dimension()


## 8 · Generate embeddings with adaptive batch size

In [ ]:
def adaptive_batch_size(n: int, min_bs: int = 8, max_bs: int = 128) -> int:
    """
    Compute batch size proportional to corpus size.
      n < 50   -> small batches (8-16)
      n = 380  -> ~19 (capped at max_bs)
      n > 2000 -> 100+ (capped at max_bs)
    """
    return min(max(min_bs, n // 20), max_bs)


def generate_embeddings(texts: list, model, batch_size: int = None) -> list:
    """
    Encode texts in batches. Returns list of float lists (or None on error).
    batch_size is auto-computed from corpus size if not provided.
    """
    if batch_size is None:
        batch_size = adaptive_batch_size(len(texts))

    print(f'Encoding {len(texts)} texts | batch_size={batch_size}')
    embeddings, skipped = [], []

    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        try:
            vecs = model.encode(batch, show_progress_bar=False)
            for j, vec in enumerate(vecs):
                if vec is None or len(vec) == 0:
                    skipped.append(i + j)
                    embeddings.append(None)
                else:
                    embeddings.append(vec.tolist())
        except Exception as exc:
            print(f'  Batch {i // batch_size} error: {exc}')
            embeddings.extend([None] * len(batch))
            skipped.extend(range(i, i + len(batch)))

        # Progress tick every 10 batches
        if (i // batch_size) % 10 == 0:
            done = min(i + batch_size, len(texts))
            print(f'  ... {done}/{len(texts)} encoded')

    ok = len(embeddings) - len(skipped)
    print(f'Embeddings generated: {ok}/{len(texts)}')
    if skipped:
        print(f'Skipped indices: {skipped}')
    return embeddings


embeddings_list = generate_embeddings(texts_to_embed, model)

Encoding 528 texts | batch_size=26
  ... 26/528 encoded
  ... 286/528 encoded
  ... 528/528 encoded
Embeddings generated: 528/528


## 9 · Connect to Pinecone & create index

In [ ]:
pc = Pinecone(api_key=PINECONE_API_KEY)

if INDEX_NAME not in pc.list_indexes().names():
    print(f'Creating index "{INDEX_NAME}" (dim={VECTOR_DIM}, metric={METRIC}) ...')
    pc.create_index(
        name=INDEX_NAME,
        dimension=VECTOR_DIM,
        metric=METRIC,
        spec=ServerlessSpec(cloud=CLOUD, region=REGION),
    )
    # Wait until ready
    while not pc.describe_index(INDEX_NAME).status['ready']:
        print('  Waiting for index to be ready...')
        time.sleep(3)
    print('  Index ready.')
else:
    print(f'Index "{INDEX_NAME}" already exists — connecting.')

index = pc.Index(INDEX_NAME)

stats = index.describe_index_stats()
print(f'Index stats: {stats.total_vector_count} vectors, dim={stats.dimension}')

Index "legis" already exists — connecting.
Index stats: 4653 vectors, dim=512


## 10 · Prepare vectors (n8n/LangChain metadata schema)


In [ ]:
def make_vector_id(text: str, idx: int) -> str:
    """Collision-resistant ID: md5(content_prefix + index + uuid)."""
    unique = f'{text[:200]}_{idx}_{time.time()}_{uuid.uuid4()}'
    return hashlib.md5(unique.encode()).hexdigest()


def prepare_vectors(metadata_list: list, embeddings: list) -> list:
    """
    Merge records + embeddings into Pinecone upsert format.
    Uses the n8n/LangChain schema: pageContent in metadata, not as a top-level field.
    Pinecone stores everything under 'metadata'; pageContent is included there
    so the LangChain retriever can reconstruct Document objects correctly.
    """
    vectors, skipped = [], 0
    for idx, (record, emb) in enumerate(zip(metadata_list, embeddings)):
        if emb is None:
            skipped += 1
            continue
        # Store pageContent inside metadata so n8n VectorStore node reads it back
        pinecone_meta = {
            **record['metadata'],
            'pageContent': record['pageContent'][:1000],  # Pinecone metadata limit
        }
        vectors.append({
            'id'      : make_vector_id(record['pageContent'], idx),
            'values'  : emb,
            'metadata': pinecone_meta,
        })
    print(f'Vectors prepared: {len(vectors)}  |  skipped: {skipped}')
    return vectors


vectors_to_upload = prepare_vectors(metadata_list, embeddings_list)

if vectors_to_upload:
    s = vectors_to_upload[0]
    print(f'\nSample vector:')
    print(f'  id            : {s["id"]}')
    print(f'  vector length : {len(s["values"])}')
    print(f'  metadata keys : {list(s["metadata"].keys())}')
    for k, v in s['metadata'].items():
        if k != 'pageContent':
            print(f'  {k:20s}: {v}')
        else:
            print(f'  {k:20s}: {str(v)[:60]}...')


Vectors prepared: 528  |  skipped: 0

Sample vector:
  id            : 3a73ef8e272a061d9f784cf30cc1e9ba
  vector length : 512
  metadata keys : ['blobType', 'date_created', 'department', 'filename', 'link', 'load_date', 'pageContent']
  blobType            : application/pdf
  date_created        : 2026-05-14T10:02:12.032Z
  department          : legis
  filename            : trudkod.gdoc
  link                : 
  load_date           : 2026-05-14
  pageContent         : Статья 1. Цели и задачи трудового законодательства
Целями т...


## 11 · Upload to Pinecone (adaptive batches, exponential-backoff retry)

In [ ]:
def batched(iterable, n):
    it = iter(iterable)
    while chunk := list(islice(it, n)):
        yield chunk


def upload_vectors(vectors: list, index, batch_size: int = None,
                   max_retries: int = 3) -> dict:
    """
    Upload vectors in batches with exponential-backoff retry.
    Returns a summary dict.
    """
    if batch_size is None:
        batch_size = adaptive_batch_size(len(vectors), min_bs=16, max_bs=100)

    print(f'Uploading {len(vectors)} vectors | batch_size={batch_size}')
    ok_b, fail_b, total_up = 0, 0, 0

    for batch_num, batch in enumerate(batched(vectors, batch_size)):
        for attempt in range(max_retries):
            try:
                index.upsert(vectors=batch, namespace=Namespace)
                ok_b      += 1
                total_up  += len(batch)
                print(f'  Batch {batch_num + 1}: OK  ({len(batch)} vectors)')
                break
            except Exception as exc:
                wait = 2 ** attempt
                if attempt == max_retries - 1:
                    fail_b += 1
                    print(f'  Batch {batch_num + 1}: FAILED after {max_retries} attempts')
                    with open(f'failed_batch_{batch_num}.txt', 'w') as fh:
                        for v in batch:
                            fh.write(f"{v['id']}\t{v['metadata'].get('article_ref', '')}\n")
                else:
                    print(f'  Batch {batch_num + 1}: retry {attempt+1}/{max_retries} in {wait}s...')
                    time.sleep(wait)

        if batch_num > 0 and batch_num % 10 == 0:
            time.sleep(0.2)   # light throttle

    total_b = ok_b + fail_b
    rate    = ok_b / total_b * 100 if total_b else 0
    summary = {'total_vectors': len(vectors), 'uploaded': total_up,
               'ok_batches': ok_b, 'fail_batches': fail_b,
               'success_rate_pct': round(rate, 1)}

    print('\n=== UPLOAD SUMMARY ===')
    for k, v in summary.items():
        print(f'  {k:22s}: {v}')
    return summary


if vectors_to_upload:
    upload_summary = upload_vectors(vectors_to_upload, index)
else:
    print('Nothing to upload — check article parsing in Step 5.')

Uploading 528 vectors | batch_size=26
  Batch 1: OK  (26 vectors)
  Batch 2: OK  (26 vectors)
  Batch 3: OK  (26 vectors)
  Batch 4: OK  (26 vectors)
  Batch 5: OK  (26 vectors)
  Batch 6: OK  (26 vectors)
  Batch 7: OK  (26 vectors)
  Batch 8: OK  (26 vectors)
  Batch 9: OK  (26 vectors)
  Batch 10: OK  (26 vectors)
  Batch 11: OK  (26 vectors)
  Batch 12: OK  (26 vectors)
  Batch 13: OK  (26 vectors)
  Batch 14: OK  (26 vectors)
  Batch 15: OK  (26 vectors)
  Batch 16: OK  (26 vectors)
  Batch 17: OK  (26 vectors)
  Batch 18: OK  (26 vectors)
  Batch 19: OK  (26 vectors)
  Batch 20: OK  (26 vectors)
  Batch 21: OK  (8 vectors)

=== UPLOAD SUMMARY ===
  total_vectors         : 528
  uploaded              : 528
  ok_batches            : 21
  fail_batches          : 0
  success_rate_pct      : 100.0


## 12 · Verify database state

In [ ]:
def check_index(index) -> None:
    stats = index.describe_index_stats()
    print('=== INDEX STATISTICS ===')
    print(f'  Total vectors : {stats.total_vector_count}')
    print(f'  Dimension     : {stats.dimension}')
    if stats.namespaces:
        for ns, ns_stats in stats.namespaces.items():
            print(f'  Namespace "{ns}": {ns_stats.vector_count} vectors')


check_index(index)

=== INDEX STATISTICS ===
  Total vectors : 5181
  Dimension     : 512
  Namespace "__default__": 3168 vectors
  Namespace "new2": 2013 vectors


## 13 · Quick semantic search test

In [ ]:
def semantic_search(query: str, model, index, top_k: int = 5) -> None:
    q_vec = model.encode([query])[0].tolist()
    results = index.query(vector=q_vec, top_k=top_k, include_metadata=True, namespace=Namespace)
    print(f"Query: '{query}'")
    print('-' * 60)
    for match in results.matches:
        m = match.metadata
        print(f"  Score   : {match.score:.4f}")
        print(f"  Ref     : {m.get('article_ref', 'N/A')}")
        print(f"  Title   : {m.get('article_short', '')[:70]}")
        print(f"  Preview : {m.get('text_preview', '')[:100]}")
        print()


TEST_QUERIES = [
    'график отпусков',   # maternity leave
    'травма на производстве',              # employee dismissal
    'индексация заработной платы за работу в праздники',             # working hours and rest
]

for q in TEST_QUERIES:
    semantic_search(q, model, index, top_k=3)
    print('=' * 60)

Query: 'график отпусков'
------------------------------------------------------------
  Score   : 0.3999
  Ref     : ТК РФ ст. 107
  Title   : Виды времени отдыха
  Preview : Статья 107. Виды времени отдыха
Видами времени отдыха являются:
перерывы в течение рабочего дня (с

  Score   : 0.3999
  Ref     : N/A
  Title   : 
  Preview : 

  Score   : 0.3999
  Ref     : N/A
  Title   : 
  Preview : 

Query: 'травма на производстве'
------------------------------------------------------------
  Score   : 0.2540
  Ref     : ТК РФ ст. 230.1
  Title   : Порядок регистрации и учета несчастных случаев на производстве
  Preview : Статья 230.1. Порядок регистрации и учета несчастных случаев на производстве
Каждый оформленный в у

  Score   : 0.2540
  Ref     : N/A
  Title   : 
  Preview : 

  Score   : 0.2540
  Ref     : N/A
  Title   : 
  Preview : 

Query: 'индексация заработной платы за работу в праздники'
------------------------------------------------------------
  Score   : 0.4654
  Ref     